In [1]:
from pathlib import Path
import base64
import importlib.util
import json
import os
import re
from typing import Any

import fitz
import pandas as pd

cwd = Path.cwd().resolve()
if (cwd / 'data').is_dir() and (cwd / 'notebooks').is_dir():
    BACKEND_ROOT = cwd
elif (cwd.parent / 'data').is_dir() and cwd.name == 'notebooks':
    BACKEND_ROOT = cwd.parent
else:
    raise RuntimeError(f'프로젝트 루트 또는 notebooks 폴더를 찾을 수 없습니다: {cwd}')

RAW_PDF_DIR = BACKEND_ROOT / 'data' / 'documents' / 'raw'
VISION_DIR = BACKEND_ROOT / 'data' / 'documents' / 'vision'
# OCR 출력과 분리된, 원본 PDF 직접 검수 정답셋
GOLD_PATH = BACKEND_ROOT / 'notebooks' / 'data' / '03_ocr_engine_comparison' / '03_manual_full_page_gold.json'
FIELD_GOLD_PATH = BACKEND_ROOT / 'notebooks' / 'data' / '03_ocr_engine_comparison' / '03_manual_field_gold.json'
OUTPUT_ROOT = BACKEND_ROOT / 'notebooks' / 'data' / '03_ocr_engine_comparison'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('PADDLE_PDX_CACHE_HOME', str(OUTPUT_ROOT / 'paddle_cache'))
os.environ.setdefault('MPLCONFIGDIR', str(OUTPUT_ROOT / 'matplotlib_cache'))

try:
    from dotenv import load_dotenv
    load_dotenv(BACKEND_ROOT / '.env')
except ImportError:
    pass

UPSTAGE_API_KEY = os.getenv('UPSTAGE_API_KEY')
ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')
CLAUDE_VISION_MODEL = os.getenv('CLAUDE_VISION_MODEL', 'claude-sonnet-4-6')

# API 키를 넣고 명시적으로 True로 바꿀 때만 외부 OCR을 호출합니다.
RUN_PADDLE = False
RUN_UPSTAGE = False
RUN_CLAUDE = os.getenv('RUN_CLAUDE', 'False').strip().lower() == 'true'
PADDLE_DEVICE = 'gpu:0'
os.environ.setdefault('PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK', 'True')

gold_dataset = json.loads(GOLD_PATH.read_text(encoding='utf-8'))
SAMPLES = gold_dataset['cases']
assert len(SAMPLES) == 10
print(f'비교 대상: {len(SAMPLES)}페이지')
print('Paddle 실행:', RUN_PADDLE, '| Upstage 실행:', RUN_UPSTAGE, '| Claude 실행:', RUN_CLAUDE)


비교 대상: 10페이지
Paddle 실행: False | Upstage 실행: False | Claude 실행: True


In [2]:
REQUIRED_MODULES = {
    'PaddleOCR': 'paddleocr',
    'Upstage/Claude HTTP client': 'requests',
}

dependency_status = pd.DataFrame([
    {'component': label, 'module': module, 'installed': importlib.util.find_spec(module) is not None}
    for label, module in REQUIRED_MODULES.items()
])
display(dependency_status)

print('설치 전에는 실행하지 않습니다. 설치가 필요할 경우에도 반드시 skn25 환경만 사용합니다.')
print('PaddleOCR: conda run -n skn25 python -m pip install paddleocr')
print('GPU용 PaddlePaddle 패키지는 CUDA/Paddle 호환 버전을 확인한 뒤 별도로 설치합니다.')


,component,module,installed
0,PaddleOCR,paddleocr,True
1,Upstage/Claude HTTP client,requests,True


설치 전에는 실행하지 않습니다. 설치가 필요할 경우에도 반드시 skn25 환경만 사용합니다.
PaddleOCR: conda run -n skn25 python -m pip install paddleocr
GPU용 PaddlePaddle 패키지는 CUDA/Paddle 호환 버전을 확인한 뒤 별도로 설치합니다.


In [3]:
def sample_key(sample: dict[str, Any]) -> str:
    return f"{sample['issuer']}__{Path(sample['file_name']).stem}__p{sample['page_number']:03d}"

def pdf_path(sample: dict[str, Any]) -> Path:
    return RAW_PDF_DIR / sample['issuer'] / sample['file_name']

def render_page(sample: dict[str, Any], scale: float = 2.0) -> Path:
    output_path = OUTPUT_ROOT / 'rendered_pages' / f"{sample_key(sample)}.png"
    output_path.parent.mkdir(parents=True, exist_ok=True)
    if output_path.exists():
        return output_path
    with fitz.open(pdf_path(sample)) as document:
        page = document[sample['page_number'] - 1]
        page.get_pixmap(matrix=fitz.Matrix(scale, scale), alpha=False).save(output_path)
    return output_path

def normalize_text_for_evaluation(text: str) -> str:
    text = re.sub(r'<br\s*/?>', '\n', text, flags=re.IGNORECASE)
    text = re.sub(r'[`*_>#|]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def canonical_record(sample: dict[str, Any], engine: str, page_text: str, raw_path: Path, tables: list[dict] | None = None) -> dict:
    return {
        'issuer': sample['issuer'],
        'file_name': sample['file_name'],
        'page_number': sample['page_number'],
        'engine': engine,
        'page_text': normalize_text_for_evaluation(page_text),
        'tables': tables or [],
        'raw_output_path': str(raw_path),
    }


In [4]:
PAGE_PATTERN = re.compile(r'\[PAGE (\d+)\]\n(.*?)(?=\n\n-{20,}\n\n\[PAGE |\Z)', re.DOTALL)

def read_vision_page(sample: dict[str, Any]) -> dict:
    vision_path = VISION_DIR / sample['issuer'] / f"{Path(sample['file_name']).stem}.txt"
    content = vision_path.read_text(encoding='utf-8')
    pages = {int(page): text for page, text in PAGE_PATTERN.findall(content)}
    return canonical_record(sample, 'Vision OCR', pages[sample['page_number']], vision_path)

def create_paddle_pipeline():
    from paddleocr import PPStructureV3

    return PPStructureV3(
        lang='korean',
        device=PADDLE_DEVICE,
        use_doc_orientation_classify=True,
        use_table_recognition=True,
        use_formula_recognition=False,
        use_seal_recognition=False,
    )

def run_paddle_ppstructure(sample: dict[str, Any], pipeline) -> dict:
    image_path = render_page(sample)
    output_dir = OUTPUT_ROOT / 'paddle_raw' / sample_key(sample)
    output_dir.mkdir(parents=True, exist_ok=True)
    results = list(pipeline.predict(str(image_path)))
    for result in results:
        result.save_to_json(save_path=str(output_dir))
        result.save_to_markdown(save_path=str(output_dir))
    return {
        'engine': 'PaddleOCR PP-StructureV3',
        'sample': sample,
        'raw_output_dir': str(output_dir),
        'note': 'Paddle의 JSON/Markdown 원본을 저장한 뒤 공통 page_text/tables 형식으로 변환합니다.',
    }

if RUN_PADDLE:
    paddle_pipeline = create_paddle_pipeline()
    paddle_runs = [run_paddle_ppstructure(sample, paddle_pipeline) for sample in SAMPLES]
else:
    paddle_runs = []


In [5]:
def require_env(name: str, value: str | None) -> str:
    if not value:
        raise RuntimeError(f'{name} 환경변수가 없습니다. 프로젝트 루트의 .env에만 설정하고 Git에 추가하지 마세요.')
    return value

def run_upstage_document_parse(sample: dict[str, Any]) -> dict:
    import requests

    api_key = require_env('UPSTAGE_API_KEY', UPSTAGE_API_KEY)
    output_path = OUTPUT_ROOT / 'upstage_raw' / f"{sample_key(sample)}.json"
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with pdf_path(sample).open('rb') as document:
        response = requests.post(
            'https://api.upstage.ai/v1/document-digitization',
            headers={'Authorization': f'Bearer {api_key}'},
            files={'document': document},
            data={'model': 'document-parse', 'ocr': 'force', 'base64_encoding': "['table']"},
            timeout=180,
        )
    if not response.ok:
        raise RuntimeError(f'Claude API 요청 실패 ({response.status_code}): {response.text[:1000]}')
    payload = response.json()
    output_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    return {
        'engine': 'Upstage Document Parse',
        'sample': sample,
        'raw_output_path': str(output_path),
        'note': '응답 스키마를 확인한 뒤 대상 page_number의 텍스트와 표를 공통 형식으로 변환합니다.',
    }

if RUN_UPSTAGE:
    upstage_runs = [run_upstage_document_parse(sample) for sample in SAMPLES]
else:
    upstage_runs = []

def run_claude_vision(sample: dict[str, Any]) -> dict:
    import requests

    api_key = require_env('ANTHROPIC_API_KEY', ANTHROPIC_API_KEY)
    image_path = render_page(sample)
    output_path = OUTPUT_ROOT / 'claude_raw' / f"{sample_key(sample)}.json"
    output_path.parent.mkdir(parents=True, exist_ok=True)
    image_base64 = base64.b64encode(image_path.read_bytes()).decode('ascii')
    payload = {
        'model': CLAUDE_VISION_MODEL,
        'max_tokens': 8192,
        'temperature': 0,
        'messages': [{
            'role': 'user',
            'content': [
                {'type': 'image', 'source': {'type': 'base64', 'media_type': 'image/png', 'data': image_base64}},
                {'type': 'text', 'text': '이 이미지는 카드 안내 PDF의 한 페이지입니다. 설명하거나 요약하지 말고, 보이는 모든 한국어·영문·숫자를 읽기 순서대로 Markdown 텍스트로 전사하세요. 표는 행과 열 관계가 보이도록 Markdown 표로 작성하세요. 읽을 수 없는 문자는 추측하지 말고 [판독불가]로 표시하세요.'},
            ],
        }],
    }
    response = requests.post(
        'https://api.anthropic.com/v1/messages',
        headers={'x-api-key': api_key, 'anthropic-version': '2023-06-01', 'content-type': 'application/json'},
        json=payload,
        timeout=180,
    )
    response.raise_for_status()
    raw_response = response.json()
    page_text = '\n'.join(block.get('text', '') for block in raw_response.get('content', []) if block.get('type') == 'text')
    output_path.write_text(json.dumps({'sample': sample, 'model': CLAUDE_VISION_MODEL, 'page_text': page_text, 'raw_response': raw_response}, ensure_ascii=False, indent=2), encoding='utf-8')
    return {'engine': 'Claude Vision', 'sample': sample, 'raw_output_path': str(output_path)}

if RUN_CLAUDE:
    claude_runs = [run_claude_vision(sample) for sample in SAMPLES]
else:
    claude_runs = []


In [6]:
vision_records = [read_vision_page(sample) for sample in SAMPLES]
comparison_manifest = {
    'engines': ['Vision OCR', 'PaddleOCR PP-StructureV3', 'Upstage Document Parse', 'Claude Vision'],
    'sample_count': len(SAMPLES),
    'current_stage': 'Vision은 기존 원문을 읽고, Paddle/Upstage/Claude Vision은 원본 출력 수집 후 공통 page_text adapter로 비교합니다. Claude는 렌더링한 지정 페이지 PNG를 Messages API에 전송합니다.',
    'evaluation_plan': {
        'full_text': ['CER', 'WER', 'content_recall', 'content_precision', 'word_sequence_similarity'],
        'facts': ['critical_fact_substring_match', 'critical_fact_token_coverage'],
        'fields': ['field_value_recall', 'field_key_value_exact_match', 'field_precision', 'field_recall', 'field_f1'],
        'tables': ['cell_text_precision', 'cell_text_recall', 'table_structure_metric'],
    },
    'normalization_rule': '원본 출력은 보관하고, 엔진별 adapter가 page_text/tables/fields 공통 형식으로 변환한 뒤 비교한다.',
}
(OUTPUT_ROOT / 'comparison_manifest.json').write_text(
    json.dumps(comparison_manifest, ensure_ascii=False, indent=2), encoding='utf-8'
)
print(json.dumps(comparison_manifest, ensure_ascii=False, indent=2))


{
  "engines": [
    "Vision OCR",
    "PaddleOCR PP-StructureV3",
    "Upstage Document Parse",
    "Claude Vision"
  ],
  "sample_count": 10,
  "current_stage": "Vision은 기존 원문을 읽고, Paddle/Upstage/Claude Vision은 원본 출력 수집 후 공통 page_text adapter로 비교합니다. Claude는 렌더링한 지정 페이지 PNG를 Messages API에 전송합니다.",
  "evaluation_plan": {
    "full_text": [
      "CER",
      "WER",
      "content_recall",
      "content_precision",
      "word_sequence_similarity"
    ],
    "facts": [
      "critical_fact_substring_match",
      "critical_fact_token_coverage"
    ],
    "fields": [
      "field_value_recall",
      "field_key_value_exact_match",
      "field_precision",
      "field_recall",
      "field_f1"
    ],
    "tables": [
      "cell_text_precision",
      "cell_text_recall",
      "table_structure_metric"
    ]
  },
  "normalization_rule": "원본 출력은 보관하고, 엔진별 adapter가 page_text/tables/fields 공통 형식으로 변환한 뒤 비교한다."
}


In [7]:
# 원본 PDF 직접 검수 정답셋으로 계산한 최종 3엔진 비교 결과
METRICS_PATH = OUTPUT_ROOT / 'manual_full_and_field_metrics.json'
metrics = json.loads(METRICS_PATH.read_text(encoding='utf-8'))
CLAUDE_METRICS_PATH = OUTPUT_ROOT / 'claude_manual_metrics.json'
if CLAUDE_METRICS_PATH.exists():
    claude_metrics = json.loads(CLAUDE_METRICS_PATH.read_text(encoding='utf-8'))
    metrics['summaries']['Claude Vision'] = claude_metrics['summary']
summary = pd.DataFrame(metrics['summaries']).T
display((summary * 100).round(2))
print('상세 페이지별 결과:', METRICS_PATH)
print('별도 HTML 보고서:', BACKEND_ROOT / 'docs' / '02_vision_paddle_upstage_ocr_comparison_report.html')


,cer,wer,content_recall,content_precision,sequence_similarity,field_value_recall,field_set_exact_match
Vision OCR,88.88,94.87,89.48,66.50,65.03,100.00,100.0
PaddleOCR PP-StructureV3,82.98,93.25,23.30,42.11,51.98,64.33,10.0
Upstage Document Parse,83.03,95.14,89.55,66.69,67.66,100.00,100.0
Claude Vision,84.94,96.60,89.99,65.23,73.55,100.00,100.0


상세 페이지별 결과: notebooks/data/03_ocr_engine_comparison/manual_full_and_field_metrics.json
별도 HTML 보고서: /home/sms/openclaw_file/RAIchU/docs/02_vision_paddle_upstage_ocr_comparison_report.html
